# Creators  → Bronze

- Reference data
- One-time extraction
- Raw JSON → Volume
- Volume → Bronze Delta

In [0]:
# %pip install google-api-python-client python-dotenv
# %restart_python

## 1. Create Volume and incoming folder

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS youtube_content_intelligence.bronze.vol_creators;

In [0]:
dbutils.fs.mkdirs("/Volumes/youtube_content_intelligence/bronze/vol_creators/incoming/")

## 2. Extract creator data

In [0]:
from src.extraction.creators import extract_creators

creators = extract_creators()

## 3. Write raw JSON to Volume

In [0]:
import json

volume_path = "/Volumes/youtube_content_intelligence/bronze/vol_creators/incoming/creators.json"

with open(volume_path, "w") as file:
    json.dump(creators, file, indent=2)

## 4. Verify raw JSON

In [0]:
display(dbutils.fs.ls("/Volumes/youtube_content_intelligence/bronze/vol_creators/incoming/"))

In [0]:
display(dbutils.fs.head("/Volumes/youtube_content_intelligence/bronze/vol_creators/incoming/creators.json", 1000))

## 5. Load raw JSON

In [0]:
df = (
    spark.read
    .option("multiLine", "true")
    .json("/Volumes/youtube_content_intelligence/bronze/vol_creators/incoming/creators.json")
)

In [0]:
df.printSchema()

## 6. Write to Bronze Delta

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("youtube_content_intelligence.bronze.brz_creators")

## 7. Validate Bronze table

In [0]:
%sql
SELECT *
FROM youtube_content_intelligence.bronze.brz_creators;

In [0]:
%sql
DESCRIBE TABLE youtube_content_intelligence.bronze.brz_creators;

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM youtube_content_intelligence.bronze.brz_creators;